In [ ]:
#Importing relevant libraries
import numpy as np
import pandas as pd
import random
from numpy.random import rand 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lime.lime_tabular import LimeTabularExplainer
import shap


random_state = 42
import warnings

#suppress future warnings
warnings.filterwarnings("ignore", category = FutureWarning)

In [ ]:
#Now we import out crop yeild dataset
url_data = 'Dataset/crop-yield.csv' #specifying the URL of the dataset

data = pd.read_csv(url_data)
data.head(5) #Displaying the first n rows from our dataset

In [ ]:
data.shape 
'''checking the shape of your dataset. We can see that there are 10,000 instances 
and 20 features. Although it's 19 features with a target class''';

In [ ]:
data.isnull().sum() #Checking for missing values. There are none!

In [ ]:
data.dtypes #chceking the datatypes. 5 of them are categorical values

In [ ]:
#Now we take out our target class away from the features
features = data.drop(['Crop_Yield_ton_per_hectare'], axis = 1)
target = data.loc[:,['Crop_Yield_ton_per_hectare']]

len(features.columns) #Checking number of features 

In [ ]:
#Now we apply the LabelEncoder to convert the categorical values into numeric
#List of columns to encode
categorical_cols = ['Soil_Type', 'Region','Season','Crop_Type', 'Irrigation_Type']

Label_encoder = {}
for col in categorical_cols:
    le = LabelEncoder()
    features[col] = le.fit_transform(data[col])
    Label_encoder[col] = le
features.head(4)# checkking our converted features

In [ ]:
#Now we normalize/scale our dataset
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
#visualizing our scaled features
features_scaled

In [ ]:
#Just to see the scaled features in a tabular form
features_table = pd.DataFrame(features_scaled, columns = features.columns)
features_table.head(4)

In [ ]:
#convert the target variable into a 1-D array, which is what most scikit-learn regression models expect
target = np.ravel(target)
#target

In [ ]:
#Now we spilt our dataset into train and test (80:20)
x_train, x_test, y_train, y_test = train_test_split(features_scaled,
                                                   target,
                                                   test_size = .2,
                                                   random_state = random_state)
fold = {'xt': x_train, 'yt': y_train, 'xv': x_test, 'yv':y_test}
#Chceking the distribution of dataset
print(f'Training set:{x_train.shape}')
print(f'Testing set:{x_test.shape}')
print(f'Training_Label set:{y_train.shape}')
print(f'Testing_Lable set:{y_test.shape}')

#### Now we employ our Feature Selection using Firefly Algorithm (FA) with Beta distribution (FA-Beta)

#### Beta distribution–based initialization

In [ ]:
def init_position_Beta(lb, ub, N, dim):
    X = np.zeros((N, dim), dtype=float)
    
    alpha = 2.0
    beta  = 2.0
    
    for i in range(N):
        for d in range(dim):
            r = np.random.beta(alpha, beta)  # Beta-distributed random number
            X[i, d] = lb[d] + (ub[d] - lb[d]) * r
            
    return X

In [ ]:
def binary_conversion(X, thres):
    """
    Convert continuous values in X to binary using sigmoid and threshold.

    """
    Xbin = (1 / (1 + np.exp(-X)) > thres).astype(int)
    return Xbin

In [ ]:
def boundary(x, lb, ub):
    """
making our search space to remian between 0 and 1
    """
    return np.clip(x, lb, ub)

In [ ]:
def error_rate(x, opts):
    """
Error rate
    """
    k    = opts['k']
    fold = opts['fold']
    xt   = fold['xt']
    yt   = fold['yt'].ravel()
    xv   = fold['xv']
    yv   = fold['yv'].ravel()

    # Ensure at least one feature is selected
    if np.sum(x) == 0:
        raise ValueError("No features selected in x. At least one feature must be 1.")

    # Select features
    xtrain = xt[:, x == 1]
    xvalid = xv[:, x == 1]

    # Train KNN regressor
    mdl = KNeighborsRegressor(n_neighbors=k)
    mdl.fit(xtrain, yt)

    # Predict and compute MAE
    ypred = mdl.predict(xvalid)
    mae = mean_absolute_error(yv, ypred)

    return mae

In [ ]:
def Fun(x, opts):
    """
Error rate and feature size
    """
    alpha = 0.8
    beta  = 1 - alpha

    max_feat = len(x)
    num_feat = np.sum(x == 1)

    if num_feat == 0:
        cost = 1  # Penalty if no features selected
    else:
        error = error_rate(x, opts)  # Compute MAE for selected features
        cost = alpha * error + beta * (num_feat / max_feat)

    return cost

In [ ]:
def jfs(xtrain, ytrain, opts):
    """
main body of code
    """
    # Parameters
    N        = opts['N']
    max_iter = opts['T']
    alpha    = opts.get('alpha', 1)
    beta0    = opts.get('beta0', 1)
    gamma    = opts.get('gamma', 1)
    theta    = opts.get('theta', 0.97)
    thres    = 0.6

    dim = xtrain.shape[1]

    lb = np.zeros(dim)
    ub = np.ones(dim)

    # Initialize positions and binary representation
    X    = init_position_Beta(lb, ub, N, dim)
    Xbin = binary_conversion(X, thres)

    # Evaluate initial fitness
    fit  = np.zeros(N)
    Xgb  = X[0,:].copy()
    fitG = float('inf')

    for i in range(N):
        fit[i] = Fun(Xbin[i,:], opts)
        if fit[i] < fitG:
            Xgb = X[i,:].copy()
            fitG = fit[i]

    # Store best fitness over iterations
    curve = np.zeros(max_iter)
    curve[0] = fitG
    print(f"Generation: 1, Best (FA): {fitG:.6f}")

    # Main loop
    for t in range(1, max_iter):
        alpha = alpha * theta

        # Rank fireflies by fitness
        ind = np.argsort(fit)
        X   = X[ind,:]
        fit = fit[ind]

        for i in range(N):
            for j in range(N):
                if fit[i] > fit[j]:
                    # Distance
                    r = np.linalg.norm(X[i,:] - X[j,:])
                    # Attractiveness
                    beta = beta0 * np.exp(-gamma * r**2)
                    # Move firefly
                    eps = np.random.rand(dim) - 0.5
                    X[i,:] = X[i,:] + beta * (X[j,:] - X[i,:]) + alpha * eps
                    # Boundary
                    X[i,:] = boundary(X[i,:], lb, ub)
                    # Binary conversion
                    xbin_i = binary_conversion(X[i,:].reshape(1,-1), thres)[0]
                    # Fitness
                    fit[i] = Fun(xbin_i, opts)
                    # Update global best
                    if fit[i] < fitG:
                        Xgb = X[i,:].copy()
                        fitG = fit[i]

        curve[t] = fitG
        print(f"Generation: {t+1}, Best (FA): {fitG:.6f}")

    # Final best solution
    Gbin      = binary_conversion(Xgb.reshape(1,-1), thres)[0]
    sel_index = np.where(Gbin == 1)[0]
    num_feat  = len(sel_index)

    return {'sf': sel_index, 'c': curve, 'nf': num_feat}

In [ ]:
#Now we set the general parameters
k = 5 #K value in KNN
N = 20 #Population size
T = 50 #Max iteration/generations
alpha  = 1       # constant
beta0  = 1       # light amplitude
gamma  = 1       # absorbtion coefficient
theta  = 0.97    # control alpha

opts = {'k':k, 'fold':fold, 'N':N, 'T':T, 'alpha':alpha, 'beta0':beta0, 'gamma':gamma, 'theta':theta}

In [ ]:
#Now we perform our feature selection by calling the module jfs
feature_selection_mdl = jfs(features_scaled, target, opts) #feature selection model
sf = feature_selection_mdl['sf'] #index of selected features

In [ ]:
print('The number of features Selected:', len(sf)) #Checking how many features got selected 

In [ ]:
#Model with selected features
num_train = np.size(x_train,0)#checks the number of samples slected in the x_train and assign to num_train
num_test = np.size(x_test, 0)#checks the number of samples slected in the x_test and assign to num_test
xtrain = x_train[:,sf]
ytrain = y_train.reshape(num_train)
xtest = x_test[:,sf]
ytest = y_test.reshape(num_test)

In [ ]:
# Build KNN regression model
knn_model = KNeighborsRegressor(n_neighbors=5)  # you can adjust k
knn_model.fit(xtrain, ytrain)  # train the model

# Make predictions on the test set
ypred = knn_model.predict(xtest)

# Evaluation
mae = mean_absolute_error(ytest, ypred)
rmse = np.sqrt(mean_squared_error(ytest, ypred))

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print('')
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

#### Now we use our SHAP Explanable AI to understand our predictions (Run this code only on one instance of your run).

In [ ]:
# Explain predictions using Kernel SHAP (model-agnostic)

# Your features (exclude target column)
feature_columns = [col for col in data.columns if col != 'Crop_Yield_ton_per_hectare']

# Scaled features for model
features_scaled = data[feature_columns].values

# Selected feature names (after FA)
selected_feature_names = [feature_columns[i] for i in sf]

# Small background sample (only selected features)
background = shap.sample(xtrain, 500) 

# KernelExplainer
explainer = shap.KernelExplainer(knn_model.predict, background)

# Compute SHAP values for test set
shap_values = explainer.shap_values(xtest)

# Plot summary
shap.summary_plot(shap_values, xtest, feature_names=selected_feature_names)

### END OF PROJECT